# 00 Study Area

Run locally in Jupyter from the repository root or `notebooks` folder. Outputs are saved in the local `goat_island_outputs` folder. Every notebook loads the seven project-confirmed parcels selected by `Parcel_Num` from `Park_Parcels/Park_Parcles.shp`.

In [4]:
%pip -q install earthengine-api geemap geopandas osmnx requests matplotlib pandas shapely pyogrio
import os, json, requests, numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import ee, geemap
from shapely.geometry import Polygon, Point
from IPython.display import display
from pathlib import Path

EE_PROJECT = "stalwart-glider-279103"  # Verify your team has access
FEMA_ASSET = f"projects/{EE_PROJECT}/assets/nfhl_flood_zones"  # Verify asset ID
# Locate the repository whether Jupyter starts in the root or notebooks folder.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "Park_Parcels" / "Park_Parcles.shp").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the gisc6379 repository.")
PARCEL_PATH = PROJECT_ROOT / "Park_Parcels" / "Park_Parcles.shp"
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_CRS = "EPSG:26914"  # UTM zone 14N
M_PER_MILE = 1609.344
ACRE_M2 = 4046.8564224
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized. Output directory:", OUTPUT_DIR)



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Earth Engine initialized. Output directory: C:\Users\Nishil\Documents\GitHub\goat-island-analysis\goat_island_outputs


In [5]:
# Always rebuild from the selected local parcel; do not reuse an older OSM boundary.
parcels = gpd.read_file(PARCEL_PATH)
if parcels.crs is None:
    raise ValueError("Park parcels must have a defined coordinate system.")
# Explicit project boundary: the seven parcel numbers confirmed by the project owner.
GOAT_ISLAND_PARCEL_IDS = ['65079661510030000', '65130905010020000', '65130905010030000', '65130905010050000', '65130905010060000', '65146334010100000', '65146334010140000']
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty:
    raise ValueError("No matching Goat Island project parcels.")
if selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Selected Goat Island parcels contain missing or empty geometry.")
# Parcel vertices include Z values; Earth Engine needs longitude/latitude pairs.
from shapely import force_2d
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Selected parcels must contain polygon geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs("EPSG:4326")
BOUNDARY_SOURCE = "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values"
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
study1_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(M_PER_MILE), crs=METRIC_CRS)
study5_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(5*M_PER_MILE), crs=METRIC_CRS)
preserve_area_acres = preserve_m.area.sum()/ACRE_M2
print(f"BOUNDARY: {BOUNDARY_SOURCE}; {len(selected)} parcel(s); area = {preserve_area_acres:.1f} acres")
preserve_gdf.to_file(OUTPUT_DIR / "preserve_boundary.geojson", driver="GeoJSON")
study1_m.to_crs(4326).to_file(OUTPUT_DIR / "one_mile_study_area.geojson", driver="GeoJSON")
roi = ee.Geometry(json.loads(json.dumps(preserve_gdf.geometry.iloc[0].__geo_interface__)))
roi1 = ee.Geometry(json.loads(json.dumps(study1_m.to_crs(4326).geometry.iloc[0].__geo_interface__)))


BOUNDARY: Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values; 7 parcel(s); area = 486.6 acres


## 1 — Selected park parcel boundary
The study extent is the seven project-confirmed parcels selected by `Parcel_Num` in the supplied park parcel shapefile (approximately 486.6 acres). This is the selected parcel extent; it follows the seven parcel IDs supplied for this project; it is not independently verified as the entire preserve. Review the map before interpreting acreage-dependent results.


In [6]:
# Always rebuild from the selected local parcel; do not reuse an older OSM boundary.
parcels = gpd.read_file(PARCEL_PATH)
if parcels.crs is None:
    raise ValueError("Park parcels must have a defined coordinate system.")
# Explicit project boundary: the seven parcel numbers confirmed by the project owner.
GOAT_ISLAND_PARCEL_IDS = ['65079661510030000', '65130905010020000', '65130905010030000', '65130905010050000', '65130905010060000', '65146334010100000', '65146334010140000']
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty:
    raise ValueError("No matching Goat Island project parcels.")
if selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Selected Goat Island parcels contain missing or empty geometry.")
# Parcel vertices include Z values; Earth Engine needs longitude/latitude pairs.
from shapely import force_2d
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Selected parcels must contain polygon geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs("EPSG:4326")
BOUNDARY_SOURCE = "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values"
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
study1_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(M_PER_MILE), crs=METRIC_CRS)
study5_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(5*M_PER_MILE), crs=METRIC_CRS)
preserve_area_acres = preserve_m.area.sum()/ACRE_M2
print(f"BOUNDARY: {BOUNDARY_SOURCE}; {len(selected)} parcel(s); area = {preserve_area_acres:.1f} acres")
preserve_gdf.to_file(OUTPUT_DIR / "preserve_boundary.geojson", driver="GeoJSON")
study1_m.to_crs(4326).to_file(OUTPUT_DIR / "one_mile_study_area.geojson", driver="GeoJSON")
roi = ee.Geometry(json.loads(json.dumps(preserve_gdf.geometry.iloc[0].__geo_interface__)))
roi1 = ee.Geometry(json.loads(json.dumps(study1_m.to_crs(4326).geometry.iloc[0].__geo_interface__)))


BOUNDARY: Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values; 7 parcel(s); area = 486.6 acres
